# Baseline — modelo inicial

Carga datos, construye features a nivel **Provider**, entrena XGBoost con hiperparámetros por defecto (`best_params={}`) y registra el run en MLflow.

**Un solo run padre** en MLflow envuelve el entrenamiento final (runs anidados internos según `train.py`).

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

def _find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "healthcare_fraud").exists():
            return candidate
    return start

_root = _find_repo_root(Path.cwd().resolve())
_src = _root / "src"
if _src.is_dir():
    sp = str(_src)
    if sp not in sys.path:
        sys.path.insert(0, sp)

import warnings

import mlflow
import pandas as pd

from healthcare_fraud.data import clean_dataframe, load_dataset, validate_dataframe
from healthcare_fraud.features import (
    FEATURE_COLS,
    build_features,
    prepare_train_val,
    split_providers,
)
from healthcare_fraud.models import setup_mlflow, train_model

warnings.filterwarnings("ignore")
print("Imports OK")

## 1. Carga y limpieza

In [ ]:
raw_tables = load_dataset()
print("Tablas cargadas:", {k: v.shape for k, v in raw_tables.items()})

In [ ]:
clean_tables = {}
for name, df in raw_tables.items():
    validated = validate_dataframe(df, name)
    clean_tables[name] = clean_dataframe(validated, name)

print("Tablas limpias:", {k: v.shape for k, v in clean_tables.items()})

## 2. Features

In [ ]:
feature_df = build_features(clean_tables)
print(f"Matriz de features: {feature_df.shape}")
print(f"Distribución de fraude:\n{feature_df['PotentialFraud'].value_counts()}")
feature_df.head()

In [ ]:
feature_df[FEATURE_COLS].describe().round(2)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(4, 4, figsize=(16, 12))
axes = axes.flatten()

for i, col in enumerate(FEATURE_COLS):
    feature_df[col].hist(ax=axes[i], bins=30, edgecolor="none", alpha=0.7)
    axes[i].set_title(col, fontsize=9)
    axes[i].set_xlabel("")

plt.tight_layout()
plt.suptitle("Distribución de features por proveedor", y=1.02, fontsize=12)
plt.show()

## 3. Split train / validación

In [ ]:
train_df, val_df = split_providers(feature_df)

overlap = set(train_df["Provider"]) & set(val_df["Provider"])
assert overlap == set(), f"Providers duplicados: {overlap}"

print(f"Train: {len(train_df)} providers | Val: {len(val_df)} providers")
print(f"Fraude en train: {train_df['PotentialFraud'].mean():.2%}")
print(f"Fraude en val:   {val_df['PotentialFraud'].mean():.2%}")

In [ ]:
X_train, X_val, y_train, y_val, preprocessor = prepare_train_val(train_df, val_df)
X_train_raw = train_df[FEATURE_COLS].values
X_val_raw = val_df[FEATURE_COLS].values
print(f"X_train (scaled matrix for reference): {X_train.shape} | X_val: {X_val.shape}")

## 4. Entrenamiento + MLflow

In [ ]:
setup_mlflow()

with mlflow.start_run(run_name="baseline_experiment"):
    mlflow.set_tag("notebook", "02_baseline")
    run_id, metrics = train_model(
        X_train_raw, y_train, X_val_raw, y_val, preprocessor, best_params={}
    )

print(f"Run ID: {run_id}")

In [ ]:
pd.DataFrame([metrics], index=["baseline"]).round(4)

## Conclusiones

- Baseline con XGBoost por defecto (vía `_build_classifier({}, ...)`).
- Priorizar **recall** y **ROC-AUC** en fraude minoritario.
- Siguiente: `03_experiments.ipynb` (Optuna + registry).